# short_volshock_bc_checkpoint_alignment_v1

## Goal

Continue the short-volshock line after `short_volshock_target_generation_handoff_v1`: test whether the fold-local short-volshock de-risk targets are actually learnable by a behavior-cloning actor, and convert the result into a checkpoint/objective selection score before any PPO.

## Governance

- **Canonical baseline**: `v19reg_v22_budget_usage_bull_guard_v1`.
- **Direct comparison baseline**: `state_conditioned_multi_objective_bc_screen_v1/multi_obj_floor45_cap20_riskfirst` and `short_volshock_target_generation_handoff_v1`.
- **Role**: `diagnose`.
- **Changed layer**: `model_selection`, `training_target`, `bc_warm_start`, `checkpoint_scoring`.
- **Target windows**: `wf_2022`, `wf_2023_2024`; `wf_2020` is skipped because the fold-local target rows start in 2020 and would leak that test window.
- **Acceptance**: BC global fit and short-volshock slice fit must pass; official warm-start replay must clear `+30pp` relative and `<=15%` MaxDD across both folds before any next PPO screen is allowed.
- **Promotion**: blocked in this notebook. Passing here only authorizes a later full replay / PPO handoff.

## External basis

- Qlib reproducible research/backtest workflow: https://github.com/microsoft/qlib and https://arxiv.org/abs/2009.11189
- FinRL train/validate/test separation for DRL trading: https://github.com/AI4Finance-Foundation/FinRL and https://arxiv.org/abs/2011.09607
- DAgger / behavior cloning covariate-shift caution: https://proceedings.mlr.press/v15/ross11a.html
- Risk-sensitive sequential decision framing: https://arxiv.org/abs/1312.4077
- Backtest overfitting caution for repeated model-selection trials: https://doi.org/10.21314/JCF.2016.322


In [1]:
from __future__ import annotations

import json
import math
import os
import random
import sys
from dataclasses import asdict
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
import torch
from IPython.display import Markdown, display

ROOT = Path.cwd()
while not (ROOT / "AGENTS.md").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
if not (ROOT / "AGENTS.md").exists():
    ROOT = Path("/Users/rziiiii/.openclaw/workspace")
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.soxl_quant.app.state_conditioned_objective import daily_path_metrics
from src.soxl_quant.app.state_objective_bc import (
    StateObjectiveBCSampleWeightConfig,
    StateObjectiveBCTrainingConfig,
    build_state_objective_bc_dataset_from_frame,
    evaluate_bc_actor_slice_fit,
    fit_state_objective_bc_actor_from_frame,
    score_bc_walkforward_drift,
    summarize_bc_walkforward_drift_screen,
)
from src.soxl_quant.app.state_objective_training_targets import (
    score_state_objective_checkpoint_candidates,
    summarize_state_objective_checkpoint_selection,
)
from src.soxl_quant.backtest import backtest_profile
from src.soxl_quant.compat import patch_gym_compat
from src.soxl_quant.config import load_profile
from src.soxl_quant.core.metric_reconciliation import build_warm_start_replay_segment, slice_warm_start_daily_path
from src.soxl_quant.data import build_dataset, feature_columns_for_packs, flatten_feature_frame, init_qlib
from src.soxl_quant.settings import LOOKBACK, QLIB_DATA_PATH

patch_gym_compat()

EXPERIMENT = "short_volshock_bc_checkpoint_alignment_v1"
CARD = ROOT / "runtime" / "model_selection" / "card" / EXPERIMENT
CARD.mkdir(parents=True, exist_ok=True)
SOURCE_CARD = ROOT / "runtime/model_selection/card/short_volshock_target_generation_handoff_v1"
BASE_TARGETS_PATH = ROOT / "runtime/model_selection/card/state_conditioned_multi_objective_bc_screen_v1/multi_obj_floor45_cap20_riskfirst_target_contract.csv"
SHORT_TARGETS_PATH = SOURCE_CARD / "short_volshock_de_risk_targets.csv"
FRAME_PATH = ROOT / "data/soxl_profile_balanced_rule_residual_v22.pkl"
PROFILE_NAME = "sparse_release_bc_ppo_screen_v1"

SEED = 20260526
TARGET_RELATIVE_RETURN = 30.0
TARGET_MAX_DD = 15.0
DRIFT_RELATIVE_FLOOR = -5.0
DRIFT_MAX_DD_CEILING = 2.0
GLOBAL_ACCURACY_MIN = 0.70
SHORT_SLICE_ACCURACY_MIN = 0.60
SHORT_SLICE_MIN_SAMPLES = 5
BC_TRAIN_CONFIG = StateObjectiveBCTrainingConfig(steps=300, lr=0.001, batch_size=128, seed=SEED)

WALK_FORWARD_WINDOWS = {
    "wf_2022": {
        "target_fold": "wf_2022_train",
        "train": ("2010-03-11", "2020-12-31"),
        "valid": ("2021-01-01", "2021-12-31"),
        "test": ("2022-01-01", "2022-12-31"),
    },
    "wf_2023_2024": {
        "target_fold": "wf_2023_2024_train",
        "train": ("2010-03-11", "2022-12-31"),
        "valid": ("2023-01-01", "2023-12-31"),
        "test": ("2024-01-01", "2024-12-31"),
    },
}

CANDIDATES = [
    {
        "candidate": "base_multi_obj_plain",
        "target_source": "base_multi_obj",
        "target_path": BASE_TARGETS_PATH,
        "complexity": 0,
        "sample_weight_config": None,
        "description": "The pre-short-volshock multi-objective target contract; used as the drift baseline.",
    },
    {
        "candidate": "short_volshock_plain",
        "target_source": "short_volshock",
        "target_path": SHORT_TARGETS_PATH,
        "complexity": 1,
        "sample_weight_config": None,
        "description": "Fold-local short-volshock de-risk targets with plain BC.",
    },
    {
        "candidate": "short_volshock_weighted_obj025_cb099",
        "target_source": "short_volshock",
        "target_path": SHORT_TARGETS_PATH,
        "complexity": 2,
        "sample_weight_config": StateObjectiveBCSampleWeightConfig(
            rebound_multiplier=2.0,
            downtrend_multiplier=4.0,
            ordinary_multiplier=1.0,
            objective_weight_power=0.25,
            class_balanced_beta=0.99,
            weight_cap=25.0,
        ),
        "description": "Short-volshock targets with downtrend/objective/action-tail weighting.",
    },
]
BASELINE_CANDIDATE = "base_multi_obj_plain"
EXTERNAL_SOURCES = [
    "https://github.com/microsoft/qlib",
    "https://arxiv.org/abs/2009.11189",
    "https://github.com/AI4Finance-Foundation/FinRL",
    "https://arxiv.org/abs/2011.09607",
    "https://proceedings.mlr.press/v15/ross11a.html",
    "https://arxiv.org/abs/1312.4077",
    "https://doi.org/10.21314/JCF.2016.322",
]


def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


def json_safe(value: Any) -> Any:
    if isinstance(value, dict):
        return {str(k): json_safe(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [json_safe(v) for v in value]
    if isinstance(value, (np.integer,)):
        return int(value)
    if isinstance(value, (np.bool_,)):
        return bool(value)
    if isinstance(value, (np.floating,)):
        value = float(value)
    if isinstance(value, float):
        return value if math.isfinite(value) else None
    if isinstance(value, pd.Timestamp):
        return value.isoformat()
    if isinstance(value, Path):
        return str(value)
    return value


def write_json(path: Path, payload: dict[str, Any]) -> None:
    path.write_text(json.dumps(json_safe(payload), indent=2, sort_keys=True, allow_nan=False) + "\n", encoding="utf-8")


def load_json(path: Path) -> dict[str, Any]:
    return json.loads(path.read_text(encoding="utf-8"))


def load_targets(path: Path) -> pd.DataFrame:
    frame = pd.read_csv(path, parse_dates=["datetime"])
    bool_cols = [
        "is_rebound_cluster",
        "is_downtrend_state",
        "release_gate_state",
        "drawdown_control_state",
        "sequence_risk_state",
        "consensus_reentry_state",
        "consensus_drawdown_cap_state",
        "state_conditioned_multi_objective_state",
        "short_volshock_state",
        "short_volshock_target_state",
        "risk_state",
    ]
    for col in bool_cols:
        if col not in frame.columns:
            frame[col] = False
        frame[col] = frame[col].fillna(False).astype(bool)
    if "short_volshock_risk_cap_delta" not in frame.columns:
        frame["short_volshock_risk_cap_delta"] = 0.0
    return frame


def fold_subset(targets: pd.DataFrame, spec: dict[str, Any]) -> pd.DataFrame:
    train_end = pd.Timestamp(spec["train"][1])
    subset = targets[
        targets["fold"].astype(str).eq(str(spec["target_fold"]))
        & (pd.to_datetime(targets["datetime"]) <= train_end)
    ].copy()
    return subset.sort_values("datetime").reset_index(drop=True)


def gate_loss(relative_return: float, max_dd: float) -> float:
    loss = 0.0
    loss += max(0.0, TARGET_RELATIVE_RETURN - float(relative_return))
    loss += max(0.0, float(max_dd) - TARGET_MAX_DD) * 10.0
    return float(loss)


def fit_slice_gate(slice_metrics: pd.DataFrame) -> tuple[bool, dict[str, Any]]:
    metrics = slice_metrics.set_index("slice") if not slice_metrics.empty else pd.DataFrame()
    failed: list[str] = []
    details: dict[str, Any] = {}
    if "all" in metrics.index:
        all_acc = float(metrics.loc["all", "accuracy"])
        details["all_accuracy"] = all_acc
        if not math.isfinite(all_acc) or all_acc < GLOBAL_ACCURACY_MIN:
            failed.append("all_accuracy")
    else:
        failed.append("all_slice_missing")
    if "short_volshock_target_state" in metrics.index:
        row = metrics.loc["short_volshock_target_state"]
        details["short_volshock_target_samples"] = int(row["samples"])
        details["short_volshock_target_accuracy"] = float(row["accuracy"])
        if int(row["samples"]) >= SHORT_SLICE_MIN_SAMPLES:
            if float(row["accuracy"]) < SHORT_SLICE_ACCURACY_MIN:
                failed.append("short_volshock_target_accuracy")
        else:
            failed.append("short_volshock_target_samples")
    else:
        details["short_volshock_target_samples"] = 0
        details["short_volshock_target_accuracy"] = None
        failed.append("short_volshock_target_slice_missing")
    details["failed_slice_checks"] = failed
    return len(failed) == 0, details


def max_drawdown_interval_from_daily(path: Path, *, label: str, candidate: str, fold: str) -> dict[str, Any]:
    daily = pd.read_csv(path, parse_dates=["datetime"])
    nav = pd.to_numeric(daily["strategy_nav"], errors="coerce").ffill()
    peak = nav.cummax()
    dd = nav / peak - 1.0
    trough_idx = int(dd.idxmin()) if len(dd) else 0
    peak_idx = int(nav.loc[:trough_idx].idxmax()) if len(nav.loc[:trough_idx]) else trough_idx
    interval = daily.loc[peak_idx:trough_idx].copy()
    action_col = daily.get("action", pd.Series("", index=daily.index)).astype(str).str.upper()
    return {
        "label": label,
        "candidate": candidate,
        "fold": fold,
        "path": str(path.relative_to(ROOT)),
        "peak_date": str(pd.Timestamp(daily.loc[peak_idx, "datetime"]).date()) if len(daily) else "",
        "trough_date": str(pd.Timestamp(daily.loc[trough_idx, "datetime"]).date()) if len(daily) else "",
        "max_dd": float(abs(dd.min()) * 100.0) if len(dd) else math.nan,
        "interval_days": int(len(interval)),
        "interval_avg_pos": float(pd.to_numeric(interval.get("position", pd.Series(0.0, index=interval.index)), errors="coerce").fillna(0.0).mean()) if len(interval) else math.nan,
        "interval_min_pos": float(pd.to_numeric(interval.get("position", pd.Series(0.0, index=interval.index)), errors="coerce").fillna(0.0).min()) if len(interval) else math.nan,
        "interval_max_pos": float(pd.to_numeric(interval.get("position", pd.Series(0.0, index=interval.index)), errors="coerce").fillna(0.0).max()) if len(interval) else math.nan,
        "interval_trades": int(action_col.loc[peak_idx:trough_idx].isin(["BUY", "SELL"]).sum()) if len(daily) else 0,
    }

source_summary = load_json(SOURCE_CARD / "summary.json")
source_selection = load_json(SOURCE_CARD / "selection.json")
profile = load_profile(PROFILE_NAME)
train_frame = pd.read_pickle(FRAME_PATH)
feature_columns = feature_columns_for_packs(profile.feature.packs, profile.name)
targets_by_source = {
    "base_multi_obj": load_targets(BASE_TARGETS_PATH),
    "short_volshock": load_targets(SHORT_TARGETS_PATH),
}
print({
    "experiment": EXPERIMENT,
    "profile": PROFILE_NAME,
    "source_decision": source_summary.get("decision"),
    "frame_rows": int(len(train_frame)),
    "target_sources": {k: int(len(v)) for k, v in targets_by_source.items()},
})


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


{'experiment': 'short_volshock_bc_checkpoint_alignment_v1', 'profile': 'sparse_release_bc_ppo_screen_v1', 'source_decision': 'short_volshock_target_generation_ready_no_promotion', 'frame_rows': 2973, 'target_sources': {'base_multi_obj': 1009, 'short_volshock': 1009}}


## Coverage And Candidate Setup

The base target is retained only as the drift baseline. The two short-volshock candidates must show both learnability and replay improvement before they can become a checkpoint or PPO handoff.

In [2]:
config_rows: list[dict[str, Any]] = []
coverage_rows: list[dict[str, Any]] = []
target_audit_rows: list[dict[str, Any]] = []

for item in CANDIDATES:
    cfg = item["sample_weight_config"]
    config_rows.append(
        {
            "candidate": item["candidate"],
            "target_source": item["target_source"],
            "complexity": int(item["complexity"]),
            "target_path": str(Path(item["target_path"]).relative_to(ROOT)),
            "sample_weight_config": None if cfg is None else asdict(cfg),
            "description": item["description"],
        }
    )
    targets = targets_by_source[str(item["target_source"])]
    for fold_name, spec in WALK_FORWARD_WINDOWS.items():
        subset = fold_subset(targets, spec)
        short_state = subset.get("short_volshock_state", pd.Series(False, index=subset.index)).fillna(False).astype(bool)
        short_target = subset.get("short_volshock_target_state", pd.Series(False, index=subset.index)).fillna(False).astype(bool)
        coverage_rows.append(
            {
                "candidate": item["candidate"],
                "fold": fold_name,
                "target_source": item["target_source"],
                "target_fold": spec["target_fold"],
                "target_rows": int(len(subset)),
                "unique_target_dates": int(subset["datetime"].nunique()) if len(subset) else 0,
                "short_volshock_state_rows": int(short_state.sum()),
                "short_volshock_target_rows": int(short_target.sum()),
                "train_start": spec["train"][0],
                "train_end": spec["train"][1],
                "test_start": spec["test"][0],
                "test_end": spec["test"][1],
            }
        )
        target_position = pd.to_numeric(subset.get("risk_target_position", pd.Series(0.0, index=subset.index)), errors="coerce")
        teacher_position = pd.to_numeric(subset.get("teacher_position", pd.Series(0.0, index=subset.index)), errors="coerce")
        target_audit_rows.append(
            {
                "candidate": item["candidate"],
                "fold": fold_name,
                "target_rows": int(len(subset)),
                "mean_teacher_position": float(teacher_position.mean()) if len(subset) else math.nan,
                "mean_target_position": float(target_position.mean()) if len(subset) else math.nan,
                "short_volshock_state_rows": int(short_state.sum()),
                "short_volshock_target_rows": int(short_target.sum()),
                "mean_short_volshock_target_position": float(target_position[short_target].mean()) if short_target.any() else math.nan,
                "mean_short_volshock_cap_delta": float(pd.to_numeric(subset.get("short_volshock_risk_cap_delta", pd.Series(0.0, index=subset.index)), errors="coerce").fillna(0.0)[short_target].mean()) if short_target.any() else 0.0,
            }
        )

candidate_configs = pd.DataFrame(config_rows)
coverage = pd.DataFrame(coverage_rows)
target_audit = pd.DataFrame(target_audit_rows)
candidate_configs.to_csv(CARD / "bc_candidate_configs.csv", index=False)
coverage.to_csv(CARD / "coverage.csv", index=False)
target_audit.to_csv(CARD / "target_audit.csv", index=False)
write_json(
    CARD / "coverage.json",
    {
        "experiment": EXPERIMENT,
        "profile": PROFILE_NAME,
        "source_experiment": "short_volshock_target_generation_handoff_v1",
        "source_decision": source_summary.get("decision"),
        "base_targets": str(BASE_TARGETS_PATH.relative_to(ROOT)),
        "short_targets": str(SHORT_TARGETS_PATH.relative_to(ROOT)),
        "frame_path": str(FRAME_PATH.relative_to(ROOT)),
        "windows": coverage_rows,
        "skipped_windows": [
            {
                "fold": "wf_2020",
                "reason": "short-volshock fold-local target rows start in 2020; using them would leak the wf_2020 test window",
            }
        ],
    },
)
display(candidate_configs)
display(coverage)
display(target_audit)


,candidate,target_source,complexity,target_path,sample_weight_config,description
0,base_multi_obj_plain,base_multi_obj,0,runtime/model_selection/card/state_conditioned...,None,The pre-short-volshock multi-objective target ...
1,short_volshock_plain,short_volshock,1,runtime/model_selection/card/short_volshock_ta...,None,Fold-local short-volshock de-risk targets with...
2,short_volshock_weighted_obj025_cb099,short_volshock,2,runtime/model_selection/card/short_volshock_ta...,"{'rebound_multiplier': 2.0, 'downtrend_multipl...",Short-volshock targets with downtrend/objectiv...


,candidate,fold,target_source,target_fold,target_rows,unique_target_dates,short_volshock_state_rows,short_volshock_target_rows,train_start,train_end,test_start,test_end
0,base_multi_obj_plain,wf_2022,base_multi_obj,wf_2022_train,253,253,0,0,2010-03-11,2020-12-31,2022-01-01,2022-12-31
1,base_multi_obj_plain,wf_2023_2024,base_multi_obj,wf_2023_2024_train,756,756,0,0,2010-03-11,2022-12-31,2024-01-01,2024-12-31
2,short_volshock_plain,wf_2022,short_volshock,wf_2022_train,253,253,135,133,2010-03-11,2020-12-31,2022-01-01,2022-12-31
3,short_volshock_plain,wf_2023_2024,short_volshock,wf_2023_2024_train,756,756,342,336,2010-03-11,2022-12-31,2024-01-01,2024-12-31
4,short_volshock_weighted_obj025_cb099,wf_2022,short_volshock,wf_2022_train,253,253,135,133,2010-03-11,2020-12-31,2022-01-01,2022-12-31
5,short_volshock_weighted_obj025_cb099,wf_2023_2024,short_volshock,wf_2023_2024_train,756,756,342,336,2010-03-11,2022-12-31,2024-01-01,2024-12-31


,candidate,fold,target_rows,mean_teacher_position,mean_target_position,short_volshock_state_rows,short_volshock_target_rows,mean_short_volshock_target_position,mean_short_volshock_cap_delta
0,base_multi_obj_plain,wf_2022,253,0.462764,0.351187,0,0,NaN,0.000000
1,base_multi_obj_plain,wf_2023_2024,756,0.297715,0.254977,0,0,NaN,0.000000
2,short_volshock_plain,wf_2022,253,0.462764,0.248057,135,133,0.05,0.196180
3,short_volshock_plain,wf_2023_2024,756,0.297715,0.202519,342,336,0.05,0.118031
4,short_volshock_weighted_obj025_cb099,wf_2022,253,0.462764,0.248057,135,133,0.05,0.196180
5,short_volshock_weighted_obj025_cb099,wf_2023_2024,756,0.297715,0.202519,342,336,0.05,0.118031


## BC Fit, Slice Fit, And Warm-Start Official Replay

Each candidate is fit separately per fold, then replayed with warm-start history and sliced back to the evaluation window. This keeps the replay metric source aligned with prior official warm-start screens.

In [3]:
init_qlib(QLIB_DATA_PATH)

fold_cache: dict[str, dict[str, Any]] = {}
for fold_name, spec in WALK_FORWARD_WINDOWS.items():
    dataset = build_dataset(
        train_seg=spec["train"],
        valid_seg=spec["valid"],
        test_seg=spec["test"],
        feature_packs=tuple(profile.feature.packs),
        profile_name=profile.name,
    )
    feature_full = flatten_feature_frame(dataset.handler.feature_frame).droplevel("instrument").copy()
    price_full = dataset.handler.price_frame.droplevel("instrument")["price"].sort_index()
    warm_seg = build_warm_start_replay_segment(feature_full.index, eval_seg=spec["test"], lookback=LOOKBACK)
    fold_cache[fold_name] = {"dataset": dataset, "feature_full": feature_full, "price_full": price_full, "warm_seg": warm_seg}

pd.DataFrame(
    [
        {
            "fold": fold,
            "feature_start": str(cache["feature_full"].index.min().date()),
            "feature_end": str(cache["feature_full"].index.max().date()),
            "test_start": WALK_FORWARD_WINDOWS[fold]["test"][0],
            "test_end": WALK_FORWARD_WINDOWS[fold]["test"][1],
            "warm_start": cache["warm_seg"][0],
            "warm_end": cache["warm_seg"][1],
        }
        for fold, cache in fold_cache.items()
    ]
).to_csv(CARD / "warm_start_segments.csv", index=False)

metric_rows: list[dict[str, Any]] = []
fit_rows: list[dict[str, Any]] = []
slice_rows: list[pd.DataFrame] = []
action_rows: list[pd.DataFrame] = []
daily_manifest: list[dict[str, str]] = []

for cfg_idx, item in enumerate(CANDIDATES):
    candidate = str(item["candidate"])
    targets = targets_by_source[str(item["target_source"])]
    for fold_name, spec in WALK_FORWARD_WINDOWS.items():
        fold_targets = fold_subset(targets, spec)
        if fold_targets.empty:
            metric_rows.append({"candidate": candidate, "fold": fold_name, "status": "skipped_no_targets"})
            continue
        set_seed(SEED + cfg_idx * 17 + len(fold_name))
        fit = fit_state_objective_bc_actor_from_frame(
            train_frame,
            targets=fold_targets,
            profile=profile,
            train_config=BC_TRAIN_CONFIG,
            sample_weight_config=item["sample_weight_config"],
            sample_size=0,
            rng_seed=SEED,
        )
        bc_dataset = build_state_objective_bc_dataset_from_frame(
            train_frame,
            targets=fold_targets,
            profile=profile,
            sample_size=0,
            rng_seed=SEED,
        )
        sm = evaluate_bc_actor_slice_fit(
            fit.actor,
            fit.obs,
            fit.actions,
            bc_dataset.metadata,
            slice_cols=(
                "short_volshock_target_state",
                "short_volshock_state",
                "is_downtrend_state",
                "risk_state",
                "is_rebound_cluster",
            ),
        )
        slice_gate, slice_details = fit_slice_gate(sm)
        sm.insert(0, "fold", fold_name)
        sm.insert(0, "candidate", candidate)
        slice_rows.append(sm)
        dist = fit.action_distribution.copy()
        dist.insert(0, "fold", fold_name)
        dist.insert(0, "candidate", candidate)
        action_rows.append(dist)
        fit_rows.append(
            {
                "candidate": candidate,
                "fold": fold_name,
                "target_rows": int(len(fold_targets)),
                "bc_samples": int(len(fit.actions)),
                "before_accuracy": float(fit.metrics.get("before_accuracy", math.nan)),
                "after_accuracy": float(fit.metrics.get("after_accuracy", math.nan)),
                "after_loss": float(fit.metrics.get("after_loss", math.nan)),
                "short_volshock_target_samples": int(slice_details.get("short_volshock_target_samples", 0) or 0),
                "short_volshock_target_accuracy": slice_details.get("short_volshock_target_accuracy"),
                "slice_gate": bool(slice_gate),
                "slice_failed_checks": json.dumps(slice_details.get("failed_slice_checks", []), sort_keys=True),
                "weight_mean": float(fit.metrics.get("weight_mean", math.nan)) if "weight_mean" in fit.metrics else math.nan,
                "weight_max": float(fit.metrics.get("weight_max", math.nan)) if "weight_max" in fit.metrics else math.nan,
            }
        )

        cache = fold_cache[fold_name]
        bt = backtest_profile(
            fit.actor,
            cache["feature_full"],
            cache["price_full"],
            position_cap_config=profile.position_cap,
            position_budget_config=profile.position_budget,
            execution_config=profile.execution,
            asset_idx=0,
            feature_columns=feature_columns,
            model_config=profile.model,
            mode_router_config=profile.mode_router,
            test_seg=cache["warm_seg"],
        )
        warm_daily = slice_warm_start_daily_path(
            portfolio_df=bt["portfolio"],
            price_series=cache["price_full"],
            eval_seg=spec["test"],
            position_history=bt.get("position_history", []),
            trade_signals=bt.get("trade_signals", []),
        )
        daily_path = CARD / f"{candidate}_{fold_name}_warm_start_official_daily_path.csv"
        warm_daily.to_csv(daily_path, index=False)
        metrics = daily_path_metrics(warm_daily)
        metric_rows.append(
            {
                "candidate": candidate,
                "fold": fold_name,
                "status": "evaluated",
                "requested_start": spec["test"][0],
                "requested_end": spec["test"][1],
                "warm_start": cache["warm_seg"][0],
                "warm_end": cache["warm_seg"][1],
                "official_warm_return": float(metrics["return"]),
                "official_warm_buyhold_return": float(metrics["buy_and_hold_return"]),
                "official_warm_relative_return": float(metrics["relative_return"]),
                "official_warm_max_dd": float(metrics["max_dd"]),
                "official_warm_avg_pos": float(metrics["avg_pos"]),
                "trades": int(bt.get("trades", 0)),
                "slice_gate": bool(slice_gate),
                "full_gate": bool(float(metrics["relative_return"]) >= TARGET_RELATIVE_RETURN and float(metrics["max_dd"]) <= TARGET_MAX_DD and bool(slice_gate)),
                "gate_loss": gate_loss(float(metrics["relative_return"]), float(metrics["max_dd"])) + (0.0 if slice_gate else 20.0),
                "daily_path": str(daily_path.relative_to(ROOT)),
            }
        )
        daily_manifest.append({"candidate": candidate, "fold": fold_name, "path": str(daily_path.relative_to(ROOT))})

fit_metrics = pd.DataFrame(fit_rows).sort_values(["candidate", "fold"]).reset_index(drop=True)
bc_slice_metrics = pd.concat(slice_rows, ignore_index=True) if slice_rows else pd.DataFrame()
action_distribution = pd.concat(action_rows, ignore_index=True) if action_rows else pd.DataFrame()
warm_metrics = pd.DataFrame(metric_rows).sort_values(["candidate", "fold"]).reset_index(drop=True)
fit_metrics.to_csv(CARD / "bc_fit_metrics.csv", index=False)
bc_slice_metrics.to_csv(CARD / "bc_slice_metrics.csv", index=False)
action_distribution.to_csv(CARD / "bc_action_distribution.csv", index=False)
warm_metrics.to_csv(CARD / "warm_start_official_metrics.csv", index=False)
pd.DataFrame(daily_manifest).to_csv(CARD / "daily_manifest.csv", index=False)
display(fit_metrics)
display(bc_slice_metrics)
display(warm_metrics)


[99019:MainThread](2026-05-26 20:18:30,391) INFO - qlib.Initialization - [config.py:452] - default_conf: client.


[99019:MainThread](2026-05-26 20:18:31,261) INFO - qlib.Initialization - [__init__.py:75] - qlib successfully initialized based on client settings.


[99019:MainThread](2026-05-26 20:18:31,263) INFO - qlib.Initialization - [__init__.py:77] - data_path={'__DEFAULT_FREQ': PosixPath('/Users/rziiiii/.openclaw/workspace/data/qlib/soxl')}


[99019:MainThread](2026-05-26 20:18:33,407) INFO - qlib.timer - [log.py:127] - Time cost: 0.001s | Loading data Done


[99019:MainThread](2026-05-26 20:18:33,411) INFO - qlib.timer - [log.py:127] - Time cost: 0.000s | fit & process data Done


...

/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  ret

/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  ret

,candidate,fold,target_rows,bc_samples,before_accuracy,after_accuracy,after_loss,short_volshock_target_samples,short_volshock_target_accuracy,slice_gate,slice_failed_checks,weight_mean,weight_max
0,base_multi_obj_plain,wf_2022,253,253,0.458498,0.849802,1.053496,0,NaN,False,"[""short_volshock_target_samples""]",NaN,NaN
1,base_multi_obj_plain,wf_2023_2024,756,755,0.329801,0.675497,1.228315,0,NaN,False,"[""all_accuracy"", ""short_volshock_target_samples""]",NaN,NaN
2,short_volshock_plain,wf_2022,253,253,0.272727,0.924901,0.979606,133,1.000000,True,[],NaN,NaN
3,short_volshock_plain,wf_2023_2024,756,755,0.181457,0.772185,1.135113,335,0.925373,True,[],NaN,NaN
4,short_volshock_weighted_obj025_cb099,wf_2022,253,253,0.272727,0.932806,0.971089,133,1.000000,True,[],1.0,5.817373
5,short_volshock_weighted_obj025_cb099,wf_2023_2024,756,755,0.181457,0.756291,1.149813,335,0.937313,True,[],1.0,2.064198


,candidate,fold,slice,samples,accuracy,loss,label_unique_actions,predicted_unique_actions,label_dominant_action,label_dominant_share,predicted_dominant_action,predicted_dominant_share
0,base_multi_obj_plain,wf_2022,all,253,0.849802,1.053496,5,4,4,0.391304,4,0.466403
1,base_multi_obj_plain,wf_2022,is_downtrend_state,122,0.811475,1.088845,4,4,3,0.352459,1,0.409836
2,base_multi_obj_plain,wf_2022,is_rebound_cluster,86,0.860465,1.038677,5,4,3,0.500000,3,0.558140
3,base_multi_obj_plain,wf_2022,risk_state,0,NaN,NaN,0,0,-1,1.000000,-1,1.000000
4,base_multi_obj_plain,wf_2022,short_volshock_state,0,NaN,NaN,0,0,-1,1.000000,-1,1.000000
5,base_multi_obj_plain,wf_2022,short_volshock_target_state,0,NaN,NaN,0,0,-1,1.000000,-1,1.000000
6,base_multi_obj_plain,wf_2023_2024,all,755,0.675497,1.228315,5,4,1,0.292715,1,0.392053
7,base_multi_obj_plain,wf_2023_2024,is_downtrend_state,480,0.654167,1.248700,4,4,1,0.397917,1,0.487500
8,base_multi_obj_plain,wf_2023_2024,is_rebound_cluster,100,0.510000,1.377382,5,4,3,0.430000,4,0.410000
9,base_multi_obj_plain,wf_2023_2024,risk_state,0,NaN,NaN,0,0,-1,1.000000,-1,1.000000


,candidate,fold,status,requested_start,requested_end,warm_start,warm_end,official_warm_return,official_warm_buyhold_return,official_warm_relative_return,official_warm_max_dd,official_warm_avg_pos,trades,slice_gate,full_gate,gate_loss,daily_path
0,base_multi_obj_plain,wf_2022,evaluated,2022-01-01,2022-12-31,2021-10-07,2022-12-30,-34.652822,-86.588072,51.935250,36.332029,0.181179,47,False,False,233.320291,runtime/model_selection/card/short_volshock_bc...
1,base_multi_obj_plain,wf_2023_2024,evaluated,2024-01-01,2024-12-31,2023-10-05,2024-12-31,33.402622,-0.356635,33.759257,20.907231,0.253076,47,False,False,79.072310,runtime/model_selection/card/short_volshock_bc...
2,short_volshock_plain,wf_2022,evaluated,2022-01-01,2022-12-31,2021-10-07,2022-12-30,-18.963676,-86.588072,67.624396,22.261989,0.097005,42,True,False,72.619890,runtime/model_selection/card/short_volshock_bc...
3,short_volshock_plain,wf_2023_2024,evaluated,2024-01-01,2024-12-31,2023-10-05,2024-12-31,4.662264,-0.356635,5.018899,16.693087,0.183114,49,True,False,41.911976,runtime/model_selection/card/short_volshock_bc...
4,short_volshock_weighted_obj025_cb099,wf_2022,evaluated,2022-01-01,2022-12-31,2021-10-07,2022-12-30,-20.715146,-86.588072,65.872926,24.745906,0.077558,42,True,False,97.459064,runtime/model_selection/card/short_volshock_bc...
5,short_volshock_weighted_obj025_cb099,wf_2023_2024,evaluated,2024-01-01,2024-12-31,2023-10-05,2024-12-31,8.466416,-0.356635,8.823051,12.799187,0.144384,45,True,False,21.176949,runtime/model_selection/card/short_volshock_bc...


## Walk-Forward Drift And Checkpoint Objective Ranking

The drift screen compares every candidate against the base multi-objective target fit. The checkpoint ranking then penalizes target-alignment loss, walk-forward gate gaps, missing short-volshock target coverage, and drawdown violations.

In [4]:
evaluated = warm_metrics[warm_metrics["status"].eq("evaluated")].copy()
fit_by_key = {(row["candidate"], row["fold"]): row for row in fit_metrics.to_dict("records")}
baseline_by_fold = {
    row["fold"]: row
    for row in evaluated[evaluated["candidate"].eq(BASELINE_CANDIDATE)].to_dict("records")
}
score_rows: list[dict[str, Any]] = []
for row in evaluated.to_dict("records"):
    baseline = baseline_by_fold.get(row["fold"])
    if baseline is None:
        continue
    scored = score_bc_walkforward_drift(
        str(row["fold"]),
        candidate_metrics={
            "relative_return": row["official_warm_relative_return"],
            "max_dd": row["official_warm_max_dd"],
            "avg_pos": row["official_warm_avg_pos"],
        },
        baseline_metrics={
            "relative_return": baseline["official_warm_relative_return"],
            "max_dd": baseline["official_warm_max_dd"],
            "avg_pos": baseline["official_warm_avg_pos"],
        },
        slice_gate=bool(row.get("slice_gate", False)),
        target_relative_return_pp=TARGET_RELATIVE_RETURN,
        target_max_dd_pct=TARGET_MAX_DD,
        relative_drift_floor_pp=DRIFT_RELATIVE_FLOOR,
        max_dd_drift_ceiling_pp=DRIFT_MAX_DD_CEILING,
    )
    fit_row = fit_by_key.get((row["candidate"], row["fold"]), {})
    scored.update(
        {
            "candidate": row["candidate"],
            "status": row["status"],
            "official_warm_return": row["official_warm_return"],
            "official_warm_buyhold_return": row["official_warm_buyhold_return"],
            "trades": int(row["trades"]),
            "after_accuracy": float(fit_row.get("after_accuracy", math.nan)),
            "short_volshock_target_samples": int(fit_row.get("short_volshock_target_samples", 0) or 0),
            "short_volshock_target_accuracy": fit_row.get("short_volshock_target_accuracy"),
            "slice_failed_checks": fit_row.get("slice_failed_checks", "[]"),
            "failed_checks_json": json.dumps(scored["failed_checks"], sort_keys=True),
            "gate_loss": float(row["gate_loss"]),
            "daily_path": row["daily_path"],
        }
    )
    score_rows.append(scored)

drift_scores = pd.DataFrame(score_rows).sort_values(["candidate", "fold"]).reset_index(drop=True)
drift_scores.to_csv(CARD / "bc_drift_scores.csv", index=False)

summary_rows: list[dict[str, Any]] = []
required_folds = tuple(WALK_FORWARD_WINDOWS.keys())
for candidate, group in drift_scores.groupby("candidate", sort=True):
    summary = summarize_bc_walkforward_drift_screen(group, required_folds=required_folds)
    summary_rows.append(
        {
            "candidate": candidate,
            "decision": summary["decision"],
            "fold_gate_pass_count": summary["fold_gate_pass_count"],
            "candidate_gate_pass_count": summary["candidate_gate_pass_count"],
            "baseline_gate_pass_count": summary["baseline_gate_pass_count"],
            "drift_gate_pass_count": summary["drift_gate_pass_count"],
            "slice_gate_pass_count": summary["slice_gate_pass_count"],
            "failed_folds_json": json.dumps(summary["failed_folds"], sort_keys=True),
            "gate_loss": float(group["gate_loss"].sum()) if "gate_loss" in group else math.inf,
            "min_relative_return": float(group["candidate_relative_return"].min()),
            "max_max_dd": float(group["candidate_max_dd"].max()),
            "mean_avg_pos": float(group["candidate_avg_pos"].mean()),
            "max_relative_delta_vs_baseline": float(group["relative_delta_vs_baseline"].max()),
            "max_dd_delta_vs_baseline": float(group["max_dd_delta_vs_baseline"].max()),
            "mean_short_volshock_accuracy": float(pd.to_numeric(group["short_volshock_target_accuracy"], errors="coerce").mean()),
            "short_volshock_target_samples": int(pd.to_numeric(group["short_volshock_target_samples"], errors="coerce").fillna(0).sum()),
            "promotion_allowed": False,
            "next_mechanism": summary["next_mechanism"],
        }
    )

candidate_screen = pd.DataFrame(summary_rows).sort_values(
    ["fold_gate_pass_count", "candidate_gate_pass_count", "drift_gate_pass_count", "gate_loss", "candidate"],
    ascending=[False, False, False, True, True],
).reset_index(drop=True)
candidate_screen.to_csv(CARD / "bc_candidate_screen.csv", index=False)

alignment_rows: list[dict[str, Any]] = []
for row in candidate_screen.to_dict("records"):
    candidate = str(row["candidate"])
    group = drift_scores[drift_scores["candidate"].eq(candidate)].copy()
    coverage_group = coverage[coverage["candidate"].eq(candidate)].copy()
    short_samples = float(row.get("short_volshock_target_samples", 0) or 0)
    short_acc = float(row.get("mean_short_volshock_accuracy", math.nan))
    accuracy_loss = max(0.0, 1.0 - float(pd.to_numeric(group["after_accuracy"], errors="coerce").mean())) if not group.empty else 1.0
    short_loss = max(0.0, SHORT_SLICE_ACCURACY_MIN - short_acc) if math.isfinite(short_acc) else 1.0
    short_missing = 0.0 if short_samples >= SHORT_SLICE_MIN_SAMPLES * len(required_folds) else 1.0
    window_gate = bool(int(row["fold_gate_pass_count"]) == len(required_folds))
    coverage_gate = bool(
        len(group) == len(required_folds)
        and bool((coverage_group["target_rows"] > 0).all())
        and short_missing == 0.0
    )
    alignment_rows.append(
        {
            "candidate": candidate,
            "alignment_loss": float(accuracy_loss + short_loss + short_missing),
            "window_gate_loss": float(row["gate_loss"]),
            "missing_target_ratio": float(short_missing),
            "window_gate": window_gate,
            "coverage_gate": coverage_gate,
            "rebound_position_mae": 0.0,
            "risk_cap_violation": float(max(0.0, row["max_max_dd"] - TARGET_MAX_DD) / TARGET_MAX_DD),
            "score": 0.0,
            "mean_bc_accuracy": float(pd.to_numeric(group["after_accuracy"], errors="coerce").mean()) if not group.empty else math.nan,
            "mean_short_volshock_accuracy": short_acc if math.isfinite(short_acc) else math.nan,
            "short_volshock_target_samples": int(short_samples),
            "fold_gate_pass_count": int(row["fold_gate_pass_count"]),
            "candidate_gate_pass_count": int(row["candidate_gate_pass_count"]),
            "max_max_dd": float(row["max_max_dd"]),
            "min_relative_return": float(row["min_relative_return"]),
        }
    )

alignment_inputs = pd.DataFrame(alignment_rows)
checkpoint_scores = score_state_objective_checkpoint_candidates(alignment_inputs)
checkpoint_summary = summarize_state_objective_checkpoint_selection(checkpoint_scores)
alignment_inputs.to_csv(CARD / "checkpoint_alignment_inputs.csv", index=False)
checkpoint_scores.to_csv(CARD / "checkpoint_objective_scores.csv", index=False)
write_json(CARD / "checkpoint_summary.json", checkpoint_summary)

display(drift_scores)
display(candidate_screen)
display(checkpoint_scores)
display(pd.DataFrame([checkpoint_summary]))


,fold,candidate_relative_return,candidate_max_dd,candidate_avg_pos,baseline_relative_return,baseline_max_dd,baseline_avg_pos,relative_delta_vs_baseline,max_dd_delta_vs_baseline,candidate_gate,...,official_warm_return,official_warm_buyhold_return,trades,after_accuracy,short_volshock_target_samples,short_volshock_target_accuracy,slice_failed_checks,failed_checks_json,gate_loss,daily_path
0,wf_2022,51.935250,36.332029,0.181179,51.935250,36.332029,0.181179,0.000000,0.000000,False,...,-34.652822,-86.588072,47,0.849802,0,NaN,"[""short_volshock_target_samples""]","[""candidate_max_dd"", ""slice_gate""]",233.320291,runtime/model_selection/card/short_volshock_bc...
1,wf_2023_2024,33.759257,20.907231,0.253076,33.759257,20.907231,0.253076,0.000000,0.000000,False,...,33.402622,-0.356635,47,0.675497,0,NaN,"[""all_accuracy"", ""short_volshock_target_samples""]","[""candidate_max_dd"", ""slice_gate""]",79.072310,runtime/model_selection/card/short_volshock_bc...
2,wf_2022,67.624396,22.261989,0.097005,51.935250,36.332029,0.181179,15.689146,-14.070040,False,...,-18.963676,-86.588072,42,0.924901,133,1.000000,[],"[""candidate_max_dd""]",72.619890,runtime/model_selection/card/short_volshock_bc...
3,wf_2023_2024,5.018899,16.693087,0.183114,33.759257,20.907231,0.253076,-28.740358,-4.214144,False,...,4.662264,-0.356635,49,0.772185,335,0.925373,[],"[""candidate_relative_return"", ""candidate_max_d...",41.911976,runtime/model_selection/card/short_volshock_bc...
4,wf_2022,65.872926,24.745906,0.077558,51.935250,36.332029,0.181179,13.937676,-11.586123,False,...,-20.715146,-86.588072,42,0.932806,133,1.000000,[],"[""candidate_max_dd""]",97.459064,runtime/model_selection/card/short_volshock_bc...
5,wf_2023_2024,8.823051,12.799187,0.144384,33.759257,20.907231,0.253076,-24.936205,-8.108044,False,...,8.466416,-0.356635,45,0.756291,335,0.937313,[],"[""candidate_relative_return"", ""relative_return...",21.176949,runtime/model_selection/card/short_volshock_bc...


,candidate,decision,fold_gate_pass_count,candidate_gate_pass_count,baseline_gate_pass_count,drift_gate_pass_count,slice_gate_pass_count,failed_folds_json,gate_loss,min_relative_return,max_max_dd,mean_avg_pos,max_relative_delta_vs_baseline,max_dd_delta_vs_baseline,mean_short_volshock_accuracy,short_volshock_target_samples,promotion_allowed,next_mechanism
0,base_multi_obj_plain,bc_walkforward_candidate_gate_failed_no_promotion,0,0,0,2,0,"[""wf_2022"", ""wf_2023_2024""]",312.392601,33.759257,36.332029,0.217127,0.000000,0.000000,NaN,0,False,revise_training_targets_before_ppo
1,short_volshock_plain,bc_walkforward_candidate_gate_failed_no_promotion,0,0,0,1,2,"[""wf_2022"", ""wf_2023_2024""]",114.531866,5.018899,22.261989,0.140060,15.689146,-4.214144,0.962687,468,False,revise_training_targets_before_ppo
2,short_volshock_weighted_obj025_cb099,bc_walkforward_candidate_gate_failed_no_promotion,0,0,0,1,2,"[""wf_2022"", ""wf_2023_2024""]",118.636013,8.823051,24.745906,0.110971,13.937676,-8.108044,0.968657,468,False,revise_training_targets_before_ppo


,candidate,alignment_loss,window_gate_loss,missing_target_ratio,window_gate,coverage_gate,rebound_position_mae,risk_cap_violation,score,mean_bc_accuracy,mean_short_volshock_accuracy,short_volshock_target_samples,fold_gate_pass_count,candidate_gate_pass_count,max_max_dd,min_relative_return,checkpoint_objective_loss,checkpoint_selection_score
0,short_volshock_plain,0.151457,114.531866,0.0,False,True,0.0,0.484133,0.0,0.848543,0.962687,468,0,0,22.261989,5.018899,2.265041,-2.265041
1,short_volshock_weighted_obj025_cb099,0.155451,118.636013,0.0,False,True,0.0,0.649727,0.0,0.844549,0.968657,468,0,0,24.745906,8.823051,2.641265,-2.641265
2,base_multi_obj_plain,2.237350,312.392601,1.0,False,False,0.0,1.422135,0.0,0.762650,NaN,0,0,0,36.332029,33.759257,18.205547,-18.205547


,decision,selected_candidate,selected_checkpoint_objective_loss,selected_checkpoint_selection_score,selected_window_gate,selected_coverage_gate,selected_alignment_loss,selected_window_gate_loss,selected_missing_target_ratio,next_mechanism,required_terms
0,checkpoint_objective_no_promotion,short_volshock_plain,2.265041,-2.265041,False,True,0.151457,114.531866,0.0,train_state_objective_checkpoint_or_bc,"[checkpoint_objective_alignment_loss, checkpoi..."


## Drawdown Interval Diagnostic And Decision

A candidate that fits labels but still draws down above 15% is not a training-length problem. The interval diagnostic records whether the short-volshock target reduces the drawdown path or simply moves the failure.

In [5]:
selected_non_base = candidate_screen[~candidate_screen["candidate"].eq(BASELINE_CANDIDATE)].copy()
if selected_non_base.empty:
    selected = candidate_screen.iloc[0].to_dict() if not candidate_screen.empty else {}
else:
    selected = selected_non_base.iloc[0].to_dict()
selected_candidate = str(selected.get("candidate", ""))
selected_scores = drift_scores[drift_scores["candidate"].eq(selected_candidate)].copy() if selected_candidate else pd.DataFrame()
selected_walkforward_summary = (
    summarize_bc_walkforward_drift_screen(selected_scores, required_folds=tuple(WALK_FORWARD_WINDOWS.keys()))
    if not selected_scores.empty
    else {
        "decision": "bc_walkforward_drift_no_rows",
        "promotion_allowed": False,
        "next_mechanism": "repair_short_volshock_bc_screen_before_training",
    }
)

all_selected_folds_pass = bool(selected_walkforward_summary.get("decision") == "bc_walkforward_drift_gate_passed_no_promotion")
any_selected_fold_pass = int(selected_walkforward_summary.get("fold_gate_pass_count", 0)) > 0
any_candidate_gate = int(selected_walkforward_summary.get("candidate_gate_pass_count", 0)) > 0
selected_short_acc = float(selected.get("mean_short_volshock_accuracy", math.nan)) if selected else math.nan
if all_selected_folds_pass:
    decision = "short_volshock_bc_alignment_passed_no_promotion"
    next_mechanism = "run full fixed-split plus wf_2020/wf_2022/wf_2023_2024 replay before any PPO or actor retention"
elif any_selected_fold_pass:
    decision = "short_volshock_bc_alignment_mixed_no_promotion"
    next_mechanism = "inspect failed fold short-volshock slices and checkpoint objective before PPO"
elif any_candidate_gate:
    decision = "short_volshock_bc_alignment_drawdown_drift_failed_no_promotion"
    next_mechanism = "add drawdown-drift penalty or checkpoint scoring before PPO"
elif math.isfinite(selected_short_acc) and selected_short_acc < SHORT_SLICE_ACCURACY_MIN:
    decision = "short_volshock_bc_slice_fit_failed_no_promotion"
    next_mechanism = "increase short-volshock slice weighting or revise target representation before replay/PPO"
else:
    decision = "short_volshock_bc_replay_gate_failed_no_promotion"
    next_mechanism = "use checkpoint objective loss or reward drawdown target; do not extend PPO from this BC actor"

interval_rows: list[dict[str, Any]] = []
for fold_name in WALK_FORWARD_WINDOWS:
    for label, candidate in [("selected", selected_candidate), ("baseline", BASELINE_CANDIDATE)]:
        path = CARD / f"{candidate}_{fold_name}_warm_start_official_daily_path.csv"
        if path.exists():
            interval_rows.append(max_drawdown_interval_from_daily(path, label=label, candidate=candidate, fold=fold_name))

drawdown_diag = pd.DataFrame(interval_rows).sort_values(["fold", "label"]).reset_index(drop=True)
drawdown_diag.to_csv(CARD / "drawdown_interval_diagnostic.csv", index=False)

conclusion: list[str] = []
if selected_scores.empty:
    conclusion.append("No selected short-volshock candidate had replay rows; this card cannot authorize training.")
else:
    for row in selected_scores.sort_values("fold").to_dict("records"):
        conclusion.append(
            f"{row['fold']}: {selected_candidate} relative={row['candidate_relative_return']:.2f}pp, "
            f"MaxDD={row['candidate_max_dd']:.2f}%, delta vs base rel={row['relative_delta_vs_baseline']:.2f}pp, "
            f"delta MaxDD={row['max_dd_delta_vs_baseline']:.2f}pp, slice_gate={bool(row['slice_gate'])}, fold_gate={bool(row['fold_gate'])}."
        )
    if not drawdown_diag.empty:
        selected_diag = drawdown_diag[drawdown_diag["label"].eq("selected")]
        for rec in selected_diag.to_dict("records"):
            conclusion.append(
                f"{rec['fold']} selected drawdown interval: {rec['peak_date']} to {rec['trough_date']}, "
                f"MaxDD={rec['max_dd']:.2f}%, avg_pos={rec['interval_avg_pos']:.3f}."
            )
conclusion.append("No promotion or actor retention from this card; BC fit and checkpoint scoring are only pre-PPO diagnostics.")
conclusion.append(f"Next mechanism: {next_mechanism}.")

selection = {
    "experiment": EXPERIMENT,
    "created_at": datetime.now(timezone.utc).isoformat(),
    "decision": decision,
    "role": "diagnose",
    "profile": PROFILE_NAME,
    "canonical_baseline": "v19reg_v22_budget_usage_bull_guard_v1",
    "direct_comparison_baseline": [
        "state_conditioned_multi_objective_bc_screen_v1/multi_obj_floor45_cap20_riskfirst",
        "short_volshock_target_generation_handoff_v1",
    ],
    "closest_historical_success_case": "short_volshock_official_replay_screen_v1 showed runtime short-volshock cap improved wf_2022 and near-missed wf_2023_2024",
    "closest_historical_failure_case": "short_volshock_sell_guard_official_replay_v1 showed sell-side runtime guard did not beat no-guard official replay",
    "duplicate_check": "This screen does not retune runtime guard thresholds; it tests fold-local short-volshock labels as BC/checkpoint objectives.",
    "changed_layer": ["model_selection", "training_target", "bc_warm_start", "checkpoint_scoring"],
    "target_relative_return_pp": TARGET_RELATIVE_RETURN,
    "target_max_dd_pct": TARGET_MAX_DD,
    "drift_relative_floor_pp": DRIFT_RELATIVE_FLOOR,
    "drift_max_dd_ceiling_pp": DRIFT_MAX_DD_CEILING,
    "folds_evaluated": list(WALK_FORWARD_WINDOWS.keys()),
    "wf_2020_coverage": False,
    "wf_2020_skip_reason": "short-volshock target rows start in 2020; using them would leak the wf_2020 test window",
    "selected_candidate": selected_candidate,
    "selected_summary": selected,
    "selected_walkforward_summary": selected_walkforward_summary,
    "checkpoint_summary": checkpoint_summary,
    "promotion_allowed": False,
    "ppo_screen_allowed_next": bool(all_selected_folds_pass),
    "training_allowed_from_this_card": bool(all_selected_folds_pass),
    "actor_retention": "no actor bundle retained; rerun notebook to regenerate diagnostic BC actors",
    "promotion_blocker": "BC/checkpoint alignment is not a promotion proof; fixed-split and full walk-forward replay are still required.",
    "next_mechanism": next_mechanism,
    "source_decision": source_summary.get("decision"),
    "external_sources": EXTERNAL_SOURCES,
    "sidecars": [
        "bc_candidate_configs.csv",
        "coverage.csv",
        "coverage.json",
        "target_audit.csv",
        "warm_start_segments.csv",
        "bc_fit_metrics.csv",
        "bc_slice_metrics.csv",
        "bc_action_distribution.csv",
        "warm_start_official_metrics.csv",
        "daily_manifest.csv",
        "bc_drift_scores.csv",
        "bc_candidate_screen.csv",
        "checkpoint_alignment_inputs.csv",
        "checkpoint_objective_scores.csv",
        "checkpoint_summary.json",
        "drawdown_interval_diagnostic.csv",
        "*_warm_start_official_daily_path.csv",
        "selection.json",
        "summary.json",
    ],
}
summary = {
    "experiment": EXPERIMENT,
    "created_at": selection["created_at"],
    "decision": decision,
    "selected_candidate": selected_candidate,
    "promotion_allowed": False,
    "ppo_screen_allowed_next": bool(all_selected_folds_pass),
    "training_allowed_from_this_card": bool(all_selected_folds_pass),
    "candidate_screen": candidate_screen.to_dict("records"),
    "selected_scores": selected_scores.to_dict("records"),
    "checkpoint_summary": checkpoint_summary,
    "drawdown_interval_diagnostic": drawdown_diag.to_dict("records"),
    "conclusion": conclusion,
    "next_mechanism": next_mechanism,
    "external_sources": EXTERNAL_SOURCES,
    "sidecars": selection["sidecars"],
}
write_json(CARD / "selection.json", selection)
write_json(CARD / "summary.json", summary)
print(json.dumps(json_safe(selection), indent=2, allow_nan=False))
display(drawdown_diag)


{
  "experiment": "short_volshock_bc_checkpoint_alignment_v1",
  "created_at": "2026-05-26T12:19:42.519116+00:00",
  "decision": "short_volshock_bc_replay_gate_failed_no_promotion",
  "role": "diagnose",
  "profile": "sparse_release_bc_ppo_screen_v1",
  "canonical_baseline": "v19reg_v22_budget_usage_bull_guard_v1",
  "direct_comparison_baseline": [
    "state_conditioned_multi_objective_bc_screen_v1/multi_obj_floor45_cap20_riskfirst",
    "short_volshock_target_generation_handoff_v1"
  ],
  "closest_historical_success_case": "short_volshock_official_replay_screen_v1 showed runtime short-volshock cap improved wf_2022 and near-missed wf_2023_2024",
  "closest_historical_failure_case": "short_volshock_sell_guard_official_replay_v1 showed sell-side runtime guard did not beat no-guard official replay",
  "duplicate_check": "This screen does not retune runtime guard thresholds; it tests fold-local short-volshock labels as BC/checkpoint objectives.",
  "changed_layer": [
    "model_selection"

,label,candidate,fold,path,peak_date,trough_date,max_dd,interval_days,interval_avg_pos,interval_min_pos,interval_max_pos,interval_trades
0,baseline,base_multi_obj_plain,wf_2022,runtime/model_selection/card/short_volshock_bc...,2022-01-03,2022-11-03,36.332029,212,0.156245,0.043688,0.477036,31
1,selected,short_volshock_plain,wf_2022,runtime/model_selection/card/short_volshock_bc...,2022-01-03,2022-11-03,22.261989,212,0.086226,0.000000,0.459725,27
2,baseline,base_multi_obj_plain,wf_2023_2024,runtime/model_selection/card/short_volshock_bc...,2024-07-10,2024-12-19,20.907231,115,0.133200,0.043620,0.464348,16
3,selected,short_volshock_plain,wf_2023_2024,runtime/model_selection/card/short_volshock_bc...,2024-03-07,2024-11-15,16.693087,177,0.140472,0.043714,0.533556,28


## Runtime Conclusion

The next cell renders the persisted decision so this notebook remains the primary single-run conclusion.

In [6]:
selection = load_json(CARD / "selection.json")
summary = load_json(CARD / "summary.json")
lines = "\n".join(f"- {line}" for line in summary["conclusion"])
checkpoint = summary.get("checkpoint_summary", {})
md = f"""
Decision: `{selection['decision']}`. Promotion is blocked.

Selected candidate: `{selection['selected_candidate']}`.

Checkpoint objective decision: `{checkpoint.get('decision')}`; selected checkpoint candidate `{checkpoint.get('selected_candidate')}` with loss `{checkpoint.get('selected_checkpoint_objective_loss')}`.

{lines}
"""
display(Markdown(md))



Decision: `short_volshock_bc_replay_gate_failed_no_promotion`. Promotion is blocked.

Selected candidate: `short_volshock_plain`.

Checkpoint objective decision: `checkpoint_objective_no_promotion`; selected checkpoint candidate `short_volshock_plain` with loss `2.2650405491687104`.

- wf_2022: short_volshock_plain relative=67.62pp, MaxDD=22.26%, delta vs base rel=15.69pp, delta MaxDD=-14.07pp, slice_gate=True, fold_gate=False.
- wf_2023_2024: short_volshock_plain relative=5.02pp, MaxDD=16.69%, delta vs base rel=-28.74pp, delta MaxDD=-4.21pp, slice_gate=True, fold_gate=False.
- wf_2022 selected drawdown interval: 2022-01-03 to 2022-11-03, MaxDD=22.26%, avg_pos=0.086.
- wf_2023_2024 selected drawdown interval: 2024-03-07 to 2024-11-15, MaxDD=16.69%, avg_pos=0.140.
- No promotion or actor retention from this card; BC fit and checkpoint scoring are only pre-PPO diagnostics.
- Next mechanism: use checkpoint objective loss or reward drawdown target; do not extend PPO from this BC actor.
